In [ ]:
import pandas as pd
import pathlib
import glob
import os
import warnings
import pyanalib.split_df_helpers as splh
from tables import NaturalNameWarning

# Suppress the NaturalNameWarning about dots in column names
warnings.filterwarnings('ignore', category=NaturalNameWarning)

def consolidate_dfs_dynamic(folder_path, output_name):
    # 1. Setup paths
    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    input_folder = pathlib.Path(folder_path)
    search_pattern = str(input_folder / f"*{output_name}*.df")
    files = sorted(glob.glob(search_pattern))

    if not files:
        print(f"No files found for: {search_pattern}")
        return None

    # 2. DISCOVER KEYS
    print(f"Inspecting first file: {os.path.basename(files[0])}")
    with pd.HDFStore(files[0], mode='r') as first_store:
        all_internal_keys = first_store.keys()
    
    keys2load = set()
    for k in all_internal_keys:
        clean_k = k.lstrip('/')
        if clean_k == "split": continue 
        base_name = clean_k.rsplit('_', 1)[0] if '_' in clean_k else clean_k
        keys2load.add(base_name)
    
    keys2load = sorted(list(keys2load))
    print(f"Found logical keys to merge: {keys2load}")

    # 3. COLLECT DATA
    master_collector = {key: [] for key in keys2load}
    for file in files:
        print(f"Reading {os.path.basename(file)}...")
        # Load all splits within this file using your helper
        file_data_dict = splh.load_dfs(file, keys2load, n_max_concat=1000)
        
        for key in keys2load:
            master_collector[key].append(file_data_dict[key])

    # 4. CONCATENATE AND SAVE
    output_file = out_dir / f"{output_name}.df"
    print(f"\nWriting to {output_file}...")

    with pd.HDFStore(output_file, mode='w') as out_store:
        for key in keys2load:
            combined = pd.concat(master_collector[key], ignore_index=False)
            out_store.put(key=key, value=combined, format="fixed")
            print(f"  - Key '{key}' consolidated: {len(combined)} rows.")
        
        out_store.put(key="split", value=pd.DataFrame({"n_split": [1]}), format="fixed")

    print("\nFile saved successfully.")
    return output_file

# --- Execution and Post-Load Verification ---

SOURCE_DIR = "/exp/sbnd/data/users/lpelegri/cafpyana/data_test"
NAME_TO_FIND = "cc1pi_5e18_CV_cc1pi_env_pandora_test_v3_create_env"

# Run the consolidation
final_path = consolidate_dfs_dynamic(SOURCE_DIR, NAME_TO_FIND)

if final_path:
    # --- VERIFICATION STEP ---
    print("\n" + "="*50)
    print("RUNNING POST-LOAD VERIFICATION")
    print("="*50)
    
    with pd.HDFStore(final_path, mode='r') as store:
        final_keys = store.keys()
        print(f"Keys found in consolidated file: {final_keys}")
        
        for k in final_keys:
            df_check = store.get(k)
            print(f"Checking Key {k:20} | Rows: {len(df_check):10} | Columns: {len(df_check.columns)}")
    
    print("="*50)
    print("Verification Complete.")